# Saddle Points in High Dimensions

Wiki reference for [Saddle Points in High Dimensions](https://ml-viz-ruby.vercel.app/wiki/saddle-points). To keep your own copy, use **File -> Save a copy in Drive**.

**The idea in one sentence.** A critical point is a local minimum only if the loss curves up in *every* direction (all Hessian eigenvalues positive) -- probability about 2^(-N) -- so in high dimensions almost every gradient-zero point is a saddle, and momentum/noise escape the plateaus that plain gradient descent crawls across.

We verify the 2^(-N) law numerically, then watch gradient descent crawl on a saddle while momentum sails through.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27', 'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a', 'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
})
rng = np.random.default_rng(0)

## 1 - The 2^(-N) argument, verified

Sample random critical points whose curvatures are +1 or -1 with equal probability in each of N directions. A point is a local minimum only if **all** N are +1.

In [ ]:
def frac_minima(N, trials=200000, rng=rng):
    curv = rng.integers(0, 2, size=(trials, N))   # 1 = curves up, 0 = curves down
    is_min = curv.all(axis=1)                      # minimum iff every direction up
    return is_min.mean()

print(f'{"N":>3} | {"empirical P(min)":>16} | {"2^-N":>10}')
for N in [1, 2, 4, 8, 12, 16]:
    emp = frac_minima(N)
    print(f'{N:>3} | {emp:>16.5f} | {2.0**-N:>10.5f}')

The empirical fraction tracks 2^(-N) and collapses fast: by N=16 fewer than 1 in 60,000 critical points is a minimum. Among the millions of parameters in a real network, genuine local minima are effectively nonexistent -- the landscape is a sea of saddles.

## 2 - A saddle in 2-D: gradient descent crawls, momentum escapes

Use `f(x, y) = x^2 - y^2`: a saddle at the origin, curving **up** along x and **down** along y (the escape direction).

In [ ]:
def f(p):  return p[0]**2 - p[1]**2
def grad(p): return np.array([2*p[0], -2*p[1]])

def optimize(momentum, steps=90, lr=0.05, beta=0.9, start=(1.5, 0.06)):
    p = np.array(start, float); v = np.zeros(2)
    traj = [p.copy()]; gnorm = [np.linalg.norm(grad(p))]
    for _ in range(steps):
        g = grad(p)
        if momentum:
            v = beta*v + g; p = p - lr*v
        else:
            p = p - lr*g
        traj.append(p.copy()); gnorm.append(np.linalg.norm(grad(p)))
    return np.array(traj), np.array(gnorm)

gd_traj, gd_g = optimize(False)
mom_traj, mom_g = optimize(True)

def escape_step(traj, thresh=1.0):
    idx = np.where(np.abs(traj[:, 1]) > thresh)[0]
    return int(idx[0]) if len(idx) else -1

print('steps to escape saddle (|y| > 1):')
print('  gradient descent:', escape_step(gd_traj))
print('  momentum        :', escape_step(mom_traj))

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.6))

# contours + trajectories
xs = np.linspace(-2, 2, 200); ys = np.linspace(-2, 2, 200)
Xg, Yg = np.meshgrid(xs, ys); Z = Xg**2 - Yg**2
a1.contourf(Xg, Yg, Z, levels=25, cmap='coolwarm', alpha=0.6)
a1.plot(gd_traj[:, 0], gd_traj[:, 1], '-o', ms=3, color='#eab308', label='gradient descent')
a1.plot(mom_traj[:, 0], mom_traj[:, 1], '-o', ms=3, color='#a5b4fc', label='momentum')
a1.scatter([0], [0], c='#e2e8f0', s=30, zorder=5, label='saddle')
a1.set_title('f(x,y) = x^2 - y^2'); a1.legend(fontsize=8)
a1.set_xlabel('x  (up-curvature)'); a1.set_ylabel('y  (escape)')

# gradient norm over iterations
a2.plot(gd_g, color='#eab308', label='gradient descent')
a2.plot(mom_g, color='#a5b4fc', label='momentum')
a2.set_yscale('log'); a2.set_xlabel('iteration'); a2.set_ylabel('gradient norm (log)')
a2.set_title('GD gradient collapses on the plateau, then recovers'); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

**What to notice.** Left: gradient descent (yellow) sweeps in along x and lingers near the origin before crawling out along y; momentum (blue) coasts straight through. Right: GD's gradient norm dips toward zero on the plateau -- the 'am I stuck?' moment -- while momentum's accumulated velocity keeps it moving. The escape route was always there; only the first-order step was slow to take it.

## 3 - Your turn

Classify a critical point from its Hessian eigenvalues, and confirm the 2^(-N) intuition. Fill in the `# TODO(you)` lines.

In [ ]:
def classify(eigs):
    eigs = np.asarray(eigs)
    # Return 'minimum' if all curvatures > 0, 'maximum' if all < 0, else 'saddle'.
    # TODO(you): implement the three-way classification
    return '???'

assert classify([2.0, 0.5, 3.1]) == 'minimum'
assert classify([-1.0, -0.2]) == 'maximum'
assert classify([2.0, -0.5]) == 'saddle'      # mixed signs -> saddle
assert classify([1.0, -3.0, 4.0, 5.0]) == 'saddle'
print('Correct - one negative curvature is enough to make it a saddle.')

<details>
<summary>Solution</summary>

```python
def classify(eigs):
    eigs = np.asarray(eigs)
    if (eigs > 0).all():
        return 'minimum'
    if (eigs < 0).all():
        return 'maximum'
    return 'saddle'
```

A single negative eigenvalue provides an escape direction, so the point is a saddle no matter how many directions curve up. That asymmetry is exactly why minima are 2^(-N)-rare.
</details>

## Key takeaways

- **A local minimum needs every direction to curve up** -- probability ~2^(-N) -- so high-dimensional loss landscapes are dominated by **saddles**, not local minima.
- **Plateaus are usually escapable saddles**, not traps: the gradient collapses, so plain GD crawls, but a negative-curvature escape direction exists.
- **Momentum accumulates velocity** along the shallow escape direction, and **SGD noise** knocks the iterate off the saddle -- both are practical saddle escapes.
- Don't diagnose a plateau as a local minimum and restart from a new seed; it is a generic feature of the landscape.

**Next:** [Convex Optimization](https://ml-viz-ruby.vercel.app/courses/optimization-ml/02-convex-optimization) for where the min = global-min guarantee holds, and [Gradient Descent Variants](https://ml-viz-ruby.vercel.app/courses/optimization-ml/01-gradient-descent-variants) for momentum and Adam.